# FedProx — Non-IID Federated Experiment, Optimized V2

This notebook runs **FedProx on the same non-IID setup** used by the current `fedavg-non-iid-optimized-v2.ipynb`.

The comparison is intentionally controlled:

- same prepared dataset
- same train/test split
- same StandardScaler
- same 39 input features and 34 classes
- same MLP: **39 → 128 → 64 → 34**
- same Adam optimizer
- same learning rate: **0.001**
- same batch size: **8192**
- same 10 clients
- same **Dirichlet label-skew partition with alpha = 0.1**
- same **3 rounds × 1 local epoch**
- same global evaluation metrics

The intended algorithmic change is only:

`FedAvg local objective` → `FedProx local objective`

FedProx adds a proximal penalty during each client's local training:

`CE_loss + (mu / 2) * ||local_weights - global_weights||²`

The server still performs sample-weighted averaging of the client models.

## FedProx setting

- `PROXIMAL_MU = 0.01`
- Keep this fixed for the first controlled comparison.
- The notebook records `mu` in every saved result so a later sensitivity study can be added without confusing runs.

## Kaggle input

Use the **same saved output of `fl-dataset-preparation-optimized-v2.ipynb`** used by both FedAvg notebooks.

1. Open this notebook on Kaggle.
2. Click **Add Input**.
3. Add the saved output of the completed FL dataset-preparation notebook.
4. Do **not** use the trained FedAvg model as input.
5. Enable a GPU (T4 or better recommended).
6. Run all cells.

The notebook independently regenerates the same deterministic non-IID partition from the common training split, seed, client count, and Dirichlet alpha. This keeps the FedAvg-Non-IID and FedProx-Non-IID comparison fair.


In [ ]:
# Cell 02: Imports
from pathlib import Path
from contextlib import nullcontext
import gc
import json
import os
import platform
import random
import time
import zipfile

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)

import torch
import torch.nn as nn


In [ ]:
# Cell 03: Configuration — aligned with FedAvg-Non-IID for a controlled comparison
EXPERIMENT_NAME = "fedprox_non_iid_optimized_v2"
EXPECTED_ARTIFACT_TYPE = "fl_prepared_iid_v2"

SEED = 42
NUM_CLIENTS = 10
NUM_ROUNDS = 3
LOCAL_EPOCHS = 1
BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 32768
LEARNING_RATE = 0.001
HIDDEN_1 = 128
HIDDEN_2 = 64

# Same non-IID partition as the FedAvg-Non-IID experiment.
DIRICHLET_ALPHA = 0.1
MIN_CLIENT_SAMPLES = 10_000
MAX_PARTITION_ATTEMPTS = 50

# FedProx hyperparameter.
PROXIMAL_MU = 0.01

USE_CLASS_WEIGHTS = False
USE_AMP = True
KEEP_TRAIN_ON_GPU = True
GPU_DATA_MAX_FRACTION = 0.70
RESUME_FROM_CHECKPOINT = True

OUTPUT_DIR = Path("/kaggle/working/fedprox_non_iid_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "fedprox_non_iid_checkpoint.pt"

print("Experiment:", EXPERIMENT_NAME)
print("Strategy: FedProx")
print("Partition: Dirichlet non-IID")
print("Dirichlet alpha:", DIRICHLET_ALPHA)
print("Proximal mu:", PROXIMAL_MU)
print("Clients:", NUM_CLIENTS)
print("Rounds:", NUM_ROUNDS)
print("Local epochs per round:", LOCAL_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)


In [ ]:
# Cell 04: Reproducibility, device, and RAM helper
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def show_ram(prefix="RAM"):
    process_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    system = psutil.virtual_memory()
    print(f"{prefix} | Process: {process_gb:.2f} GB | System used: {system.percent:.1f}%")


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_b/1024**3:.2f} GB free / {total_b/1024**3:.2f} GB total")
else:
    print("WARNING: GPU is strongly recommended for the full-data experiment.")

show_ram("Startup")


In [ ]:
# Cell 05: Automatically locate the prepared dataset artifact in Kaggle inputs
def find_prepared_artifact(root=Path("/kaggle/input")):
    matches = []
    for metadata_path in root.rglob("metadata.json"):
        try:
            with open(metadata_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            if meta.get("artifact_type") == EXPECTED_ARTIFACT_TYPE:
                matches.append((metadata_path.parent, meta))
        except Exception:
            continue

    if not matches:
        raise FileNotFoundError(
            "Prepared FL artifact not found. First run and Save Version of "
            "fl-dataset-preparation-optimized-v2.ipynb, then Add Input -> that notebook output."
        )

    if len(matches) > 1:
        print("Multiple matching prepared artifacts found; using the first one:")
        for path, _ in matches:
            print("-", path)

    return matches[0]


DATA_DIR, metadata = find_prepared_artifact()

print("Prepared artifact:", DATA_DIR)
print("Dataset fraction:", metadata["dataset_frac_requested"])
print("Train samples:", f"{metadata['train_samples']:,}")
print("Test samples:", f"{metadata['test_samples']:,}")
print("Prepared artifact partition:", metadata["partition"])
print("This notebook partition: Dirichlet non-IID")
print("Federated strategy: FedProx")

if metadata["seed"] != SEED:
    raise ValueError(f"Seed mismatch: artifact={metadata['seed']} notebook={SEED}")
if int(metadata["num_clients"]) != NUM_CLIENTS:
    raise ValueError(
        f"Client-count mismatch: artifact={metadata['num_clients']} notebook={NUM_CLIENTS}"
    )


In [ ]:
# Cell 06: Load common data, split, scaler, classes, and features
# IMPORTANT: saved IID client positions are deliberately NOT loaded.
load_start = time.perf_counter()

with open(DATA_DIR / "classes.json", "r", encoding="utf-8") as f:
    classes = json.load(f)
with open(DATA_DIR / "feature_cols.json", "r", encoding="utf-8") as f:
    feature_cols = json.load(f)

X_mm = np.load(DATA_DIR / "X.npy", mmap_mode="r")
y_mm = np.load(DATA_DIR / "y.npy", mmap_mode="r")
train_idx = np.load(DATA_DIR / "train_idx.npy")
test_idx = np.load(DATA_DIR / "test_idx.npy")
scaler_mean = np.load(DATA_DIR / "scaler_mean.npy").astype(np.float32)
scaler_scale = np.load(DATA_DIR / "scaler_scale.npy").astype(np.float32)

assert X_mm.shape[1] == 39
assert len(feature_cols) == 39
assert len(classes) == 34
assert scaler_mean.shape == (39,)
assert scaler_scale.shape == (39,)

print("X mmap:", X_mm.shape, X_mm.dtype)
print("y mmap:", y_mm.shape, y_mm.dtype)
print("Classes:", len(classes))
print("Features:", len(feature_cols))
show_ram("After mmap loading")


In [ ]:
# Cell 07: Materialize the SAME train/test arrays and apply the SAME saved StandardScaler
materialize_start = time.perf_counter()

X_train = np.ascontiguousarray(X_mm[train_idx], dtype=np.float32)
y_train = np.ascontiguousarray(y_mm[train_idx], dtype=np.int64)
X_test = np.ascontiguousarray(X_mm[test_idx], dtype=np.float32)
y_test = np.ascontiguousarray(y_mm[test_idx], dtype=np.int64)

# In-place standardization; statistics were fitted by the preparation notebook on training data only.
X_train -= scaler_mean
X_train /= scaler_scale
X_test -= scaler_mean
X_test /= scaler_scale

if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
    raise ValueError("Non-finite values found after scaling.")

# We now use positions relative to X_train/y_train, exactly as the IID notebook does.
del X_mm, y_mm, train_idx, test_idx
gc.collect()

materialize_seconds = time.perf_counter() - materialize_start
print("Materialized training samples:", f"{len(y_train):,}")
print("Materialized testing samples: ", f"{len(y_test):,}")
print("Materialize + scale time:", f"{materialize_seconds:.2f} s")
show_ram("After materialization")


In [ ]:
# Cell 08: Build deterministic non-IID client partitions with Dirichlet label skew
# Each training sample is assigned to exactly one client. No sample is dropped or duplicated.
def build_dirichlet_partitions(
    labels,
    num_clients,
    num_classes,
    alpha,
    seed,
    min_client_samples=1,
    max_attempts=50,
):
    labels = np.asarray(labels)

    for attempt in range(max_attempts):
        # A different deterministic RNG stream per attempt.
        rng = np.random.default_rng(seed + attempt)
        client_chunks = [[] for _ in range(num_clients)]

        for class_id in range(num_classes):
            class_positions = np.flatnonzero(labels == class_id).astype(np.int32, copy=False)
            rng.shuffle(class_positions)

            proportions = rng.dirichlet(np.full(num_clients, alpha, dtype=np.float64))
            counts = rng.multinomial(len(class_positions), proportions)
            boundaries = np.cumsum(counts)[:-1]
            class_splits = np.split(class_positions, boundaries)

            for client_id, split in enumerate(class_splits):
                if len(split):
                    client_chunks[client_id].append(split)

        client_positions = []
        for client_id, chunks in enumerate(client_chunks):
            if chunks:
                positions = np.concatenate(chunks).astype(np.int32, copy=False)
            else:
                positions = np.empty(0, dtype=np.int32)
            rng.shuffle(positions)
            client_positions.append(positions)

        sizes = np.array([len(p) for p in client_positions], dtype=np.int64)
        if sizes.min() >= min_client_samples:
            return client_positions, attempt + 1

    raise RuntimeError(
        f"Could not create a partition with at least {min_client_samples:,} samples/client "
        f"after {max_attempts} attempts. Increase alpha or reduce MIN_CLIENT_SAMPLES."
    )


partition_start = time.perf_counter()
client_positions, partition_attempts = build_dirichlet_partitions(
    labels=y_train,
    num_clients=NUM_CLIENTS,
    num_classes=len(classes),
    alpha=DIRICHLET_ALPHA,
    seed=SEED,
    min_client_samples=MIN_CLIENT_SAMPLES,
    max_attempts=MAX_PARTITION_ATTEMPTS,
)
partition_seconds = time.perf_counter() - partition_start

client_sizes = np.array([len(p) for p in client_positions], dtype=np.int64)
print(f"Non-IID partition created in {partition_seconds:.2f} s (attempt {partition_attempts}).")
print("Client sizes:", client_sizes.tolist())
print("Min / Max client size:", f"{client_sizes.min():,}", "/", f"{client_sizes.max():,}")


In [ ]:
# Cell 09: Verify partition integrity and quantify heterogeneity
all_positions = np.concatenate(client_positions)

assert len(all_positions) == len(y_train), "Some training samples were lost or duplicated."
assert np.array_equal(np.sort(all_positions), np.arange(len(y_train), dtype=all_positions.dtype)), (
    "Non-IID partitions are not an exact one-time cover of the training split."
)

num_classes = len(classes)
global_counts = np.bincount(y_train, minlength=num_classes).astype(np.int64)
global_probs = global_counts / global_counts.sum()


def normalized_entropy(counts):
    probs = counts[counts > 0] / counts.sum()
    entropy = -(probs * np.log(probs)).sum()
    return float(entropy / np.log(len(counts)))


def js_divergence(p, q):
    # Jensen-Shannon divergence, natural-log scale. Stable with zero-probability classes.
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    mask_p = p > 0
    mask_q = q > 0
    kl_pm = np.sum(p[mask_p] * np.log(p[mask_p] / m[mask_p]))
    kl_qm = np.sum(q[mask_q] * np.log(q[mask_q] / m[mask_q]))
    return float(0.5 * (kl_pm + kl_qm))


partition_rows = []
class_count_rows = []

for client_id, positions in enumerate(client_positions):
    counts = np.bincount(y_train[positions], minlength=num_classes).astype(np.int64)
    probs = counts / counts.sum()

    partition_rows.append({
        "client_id": client_id,
        "samples": int(len(positions)),
        "sample_fraction": float(len(positions) / len(y_train)),
        "active_classes": int(np.count_nonzero(counts)),
        "dominant_class_id": int(np.argmax(counts)),
        "dominant_class_name": classes[int(np.argmax(counts))],
        "dominant_class_share": float(probs.max()),
        "normalized_label_entropy": normalized_entropy(counts),
        "js_divergence_from_global": js_divergence(probs, global_probs),
    })

    for class_id, class_name in enumerate(classes):
        class_count_rows.append({
            "client_id": client_id,
            "class_id": class_id,
            "class_name": class_name,
            "count": int(counts[class_id]),
            "proportion": float(probs[class_id]),
        })

partition_summary_df = pd.DataFrame(partition_rows)
client_class_distribution_df = pd.DataFrame(class_count_rows)

print("All training samples assigned exactly once.")
display(partition_summary_df.round(4))
print("Mean Jensen-Shannon divergence from global distribution:",
      f"{partition_summary_df['js_divergence_from_global'].mean():.4f}")
print("Mean dominant-class share:",
      f"{partition_summary_df['dominant_class_share'].mean():.4f}")


In [ ]:
# Cell 10: Visualize the non-IID class distribution before training
pivot = client_class_distribution_df.pivot(
    index="client_id", columns="class_name", values="proportion"
).fillna(0.0)

plt.figure(figsize=(18, 6))
plt.imshow(pivot.values, aspect="auto", interpolation="nearest")
plt.colorbar(label="Within-client class proportion")
plt.xticks(np.arange(len(classes)), classes, rotation=90, fontsize=7)
plt.yticks(np.arange(NUM_CLIENTS), [f"Client {i}" for i in range(NUM_CLIENTS)])
plt.xlabel("Class")
plt.ylabel("Client")
plt.title(f"FedProx Non-IID — Dirichlet Label Distribution (alpha={DIRICHLET_ALPHA})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fedprox_non_iid_client_class_distribution.png", dpi=180)
plt.show()


In [ ]:
# Cell 11: Create tensors and choose training-data location
# Test data stays on CPU. Training data moves to GPU once if it safely fits.
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

train_bytes = X_train.nbytes + y_train.nbytes
move_train_to_gpu = False

if KEEP_TRAIN_ON_GPU and device.type == "cuda":
    free_bytes, _ = torch.cuda.mem_get_info()
    move_train_to_gpu = train_bytes < GPU_DATA_MAX_FRACTION * free_bytes

if move_train_to_gpu:
    print(f"Moving {train_bytes/1024**3:.2f} GB of training data to GPU once...")
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    train_data_device = device
else:
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    train_data_device = torch.device("cpu")

# Tensor objects now own/reference everything needed.
del X_train, y_train, X_test, y_test, all_positions
gc.collect()

print("Training data location:", train_data_device)
print("Test data location: CPU")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Cell 12: Define the exact centralized-baseline MLP
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_1=128, hidden_2=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, num_classes),
        )

    def forward(self, x):
        return self.network(x)


global_model = MLP(
    input_dim=len(feature_cols),
    num_classes=len(classes),
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
).to(device)

# Reuse one local-model object sequentially to avoid unnecessary allocations.
local_model = MLP(
    input_dim=len(feature_cols),
    num_classes=len(classes),
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
).to(device)

num_parameters = sum(p.numel() for p in global_model.parameters())
amp_enabled = bool(USE_AMP and device.type == "cuda")

print(global_model)
print("Trainable parameters:", f"{num_parameters:,}")
print("AMP enabled:", amp_enabled)


In [ ]:
# Cell 13: Loss function
if USE_CLASS_WEIGHTS:
    train_counts = np.bincount(
        y_train_t.detach().cpu().numpy(),
        minlength=len(classes),
    ).astype(np.float64)
    if np.any(train_counts == 0):
        raise ValueError("A class has zero training samples.")
    class_weights_np = len(y_train_t) / (len(classes) * train_counts)
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

print("Class weights:", USE_CLASS_WEIGHTS)
print("Loss: CrossEntropyLoss")


In [ ]:
# Cell 14: Sanity check one non-IID client batch before expensive FL training
first_nonempty_client = next(i for i, p in enumerate(client_positions) if len(p) > 0)
n = min(BATCH_SIZE, len(client_positions[first_nonempty_client]))
positions = torch.from_numpy(
    client_positions[first_nonempty_client][:n].astype(np.int64, copy=False)
).to(train_data_device)

X_batch = X_train_t[positions]
y_batch = y_train_t[positions]

if train_data_device.type == "cpu":
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

local_model.eval()
with torch.no_grad():
    ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
    with ctx:
        logits = local_model(X_batch)

print("Client:", first_nonempty_client)
print("X:", X_batch.shape, X_batch.dtype, X_batch.device)
print("y:", y_batch.shape, y_batch.dtype, y_batch.device)
print("Output:", logits.shape)
print("Finite outputs:", bool(torch.isfinite(logits).all()))

assert logits.shape == (n, len(classes))
assert torch.isfinite(logits).all()

del positions, X_batch, y_batch, logits
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Cell 15: FedProx local client training
# Each client starts from the SAME current global model.
# FedProx adds a proximal penalty that discourages the local model from
# drifting too far from those global parameters during local training.
def proximal_penalty(model, global_reference):
    penalty = torch.zeros((), device=device)
    for local_param, global_param in zip(model.parameters(), global_reference):
        penalty = penalty + torch.sum((local_param - global_param) ** 2)
    return 0.5 * PROXIMAL_MU * penalty


def train_one_client(global_state, global_reference, client_id, round_number):
    positions_np = client_positions[client_id]

    local_model.load_state_dict(global_state, strict=True)
    local_model.train()

    optimizer = torch.optim.Adam(local_model.parameters(), lr=LEARNING_RATE)
    grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    running_ce_loss = 0.0
    running_objective_loss = 0.0
    running_prox_penalty = 0.0
    correct = 0
    total = 0
    client_start = time.perf_counter()

    positions_t = torch.from_numpy(
        positions_np.astype(np.int64, copy=False)
    ).to(train_data_device)

    for local_epoch in range(1, LOCAL_EPOCHS + 1):
        # Same deterministic shuffling scheme as FedAvg-Non-IID.
        shuffle_seed = SEED + round_number * 100_000 + client_id * 1_000 + local_epoch
        generator = torch.Generator(device=train_data_device)
        generator.manual_seed(shuffle_seed)
        order = torch.randperm(
            len(positions_t),
            generator=generator,
            device=train_data_device,
        )

        num_batches = (len(positions_t) + BATCH_SIZE - 1) // BATCH_SIZE

        for batch_number, start in enumerate(
            range(0, len(positions_t), BATCH_SIZE),
            start=1,
        ):
            batch_positions = positions_t[order[start:start + BATCH_SIZE]]
            X_batch = X_train_t[batch_positions]
            y_batch = y_train_t[batch_positions]

            if train_data_device.type == "cpu":
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

            optimizer.zero_grad(set_to_none=True)

            ctx = (
                torch.amp.autocast("cuda", dtype=torch.float16)
                if amp_enabled
                else nullcontext()
            )
            with ctx:
                logits = local_model(X_batch)
                ce_loss = criterion(logits, y_batch)

            prox = proximal_penalty(local_model, global_reference)
            objective = ce_loss + prox

            grad_scaler.scale(objective).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()

            batch_n = y_batch.size(0)
            running_ce_loss += ce_loss.detach().item() * batch_n
            running_objective_loss += objective.detach().item() * batch_n
            running_prox_penalty += prox.detach().item() * batch_n
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total += batch_n

            # Sparse logging keeps notebook output small and fast.
            if batch_number % 250 == 0 or batch_number == num_batches:
                elapsed = time.perf_counter() - client_start
                print(
                    f"  client {client_id:02d} | local epoch {local_epoch}/{LOCAL_EPOCHS} | "
                    f"batch {batch_number:,}/{num_batches:,} | "
                    f"CE {running_ce_loss/total:.4f} | "
                    f"prox {running_prox_penalty/total:.5f} | "
                    f"acc {100*correct/total:.2f}% | "
                    f"{total/elapsed:,.0f} samples/s"
                )

        del order

    elapsed = time.perf_counter() - client_start
    local_state_cpu = {
        key: value.detach().cpu().clone()
        for key, value in local_model.state_dict().items()
    }

    del positions_t, optimizer, grad_scaler

    return {
        "state": local_state_cpu,
        "samples": int(len(positions_np)),
        # Keep CE loss separately so it remains comparable with FedAvg.
        "ce_loss": float(running_ce_loss / total),
        "objective_loss": float(running_objective_loss / total),
        "proximal_penalty": float(running_prox_penalty / total),
        "accuracy": float(correct / total),
        "seconds": float(elapsed),
        "samples_per_second": float(total / elapsed),
    }


In [ ]:
# Cell 16: One complete FedProx round
# FedProx changes LOCAL training; server aggregation remains sample-weighted averaging.
def run_fedprox_round(round_number):
    round_start = time.perf_counter()

    # State used to reset every local model to the same round-start weights.
    global_state = {
        key: value.detach().clone()
        for key, value in global_model.state_dict().items()
    }

    # Parameter references used by the proximal penalty.
    global_reference = [
        param.detach().clone()
        for param in global_model.parameters()
    ]

    total_samples = sum(len(x) for x in client_positions)
    aggregated_state = None
    client_rows = []

    for client_id in range(NUM_CLIENTS):
        print(
            f"\nRound {round_number}/{NUM_ROUNDS} — "
            f"training client {client_id}/{NUM_CLIENTS - 1}"
        )

        result = train_one_client(
            global_state,
            global_reference,
            client_id,
            round_number,
        )
        weight = result["samples"] / total_samples

        if aggregated_state is None:
            aggregated_state = {
                key: tensor * weight
                for key, tensor in result["state"].items()
            }
        else:
            for key in aggregated_state:
                aggregated_state[key].add_(
                    result["state"][key],
                    alpha=weight,
                )

        client_rows.append({
            "round": round_number,
            "client_id": client_id,
            "samples": result["samples"],
            "aggregation_weight": float(weight),
            "train_loss": result["ce_loss"],
            "objective_loss": result["objective_loss"],
            "proximal_penalty": result["proximal_penalty"],
            "train_accuracy": result["accuracy"],
            "seconds": result["seconds"],
            "samples_per_second": result["samples_per_second"],
        })

        print(
            f"Client {client_id:02d} complete | samples={result['samples']:,} | "
            f"weight={weight:.4f} | CE={result['ce_loss']:.4f} | "
            f"prox={result['proximal_penalty']:.5f} | "
            f"acc={100*result['accuracy']:.2f}% | "
            f"time={result['seconds']:.2f}s"
        )

        del result

    global_model.load_state_dict(aggregated_state, strict=True)

    round_seconds = time.perf_counter() - round_start
    client_df = pd.DataFrame(client_rows)

    summary = {
        "round": round_number,
        # Same column name as FedAvg: classification loss only.
        "client_train_loss": float(
            np.average(client_df["train_loss"], weights=client_df["samples"])
        ),
        "client_objective_loss": float(
            np.average(client_df["objective_loss"], weights=client_df["samples"])
        ),
        "client_proximal_penalty": float(
            np.average(client_df["proximal_penalty"], weights=client_df["samples"])
        ),
        "client_train_accuracy": float(
            np.average(client_df["train_accuracy"], weights=client_df["samples"])
        ),
        "client_train_accuracy_mean": float(client_df["train_accuracy"].mean()),
        "client_train_accuracy_std": float(client_df["train_accuracy"].std(ddof=0)),
        "client_train_accuracy_min": float(client_df["train_accuracy"].min()),
        "client_train_accuracy_max": float(client_df["train_accuracy"].max()),
        "round_training_seconds": float(round_seconds),
        "client_mean_samples_per_second": float(
            client_df["samples_per_second"].mean()
        ),
    }

    del global_state, global_reference, aggregated_state
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return summary, client_df


In [ ]:
# Cell 17: Global evaluation — same headline metrics as centralized and IID baselines
def evaluate_global_model(model):
    model.eval()
    predictions = np.empty(len(y_test_t), dtype=np.int16)
    targets = np.empty(len(y_test_t), dtype=np.int16)

    start_time = time.perf_counter()
    write_pos = 0

    with torch.no_grad():
        for start in range(0, len(y_test_t), EVAL_BATCH_SIZE):
            X_batch = X_test_t[start:start + EVAL_BATCH_SIZE].to(device)
            y_batch = y_test_t[start:start + EVAL_BATCH_SIZE].to(device)

            ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
            with ctx:
                logits = model(X_batch)

            pred = logits.argmax(dim=1)
            n = len(y_batch)
            predictions[write_pos:write_pos + n] = pred.cpu().numpy()
            targets[write_pos:write_pos + n] = y_batch.cpu().numpy()
            write_pos += n

    evaluation_seconds = time.perf_counter() - start_time

    accuracy = accuracy_score(targets, predictions)
    balanced_accuracy = balanced_accuracy_score(targets, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        targets, predictions, average="macro", zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        targets, predictions, average="weighted", zero_division=0
    )
    mcc = matthews_corrcoef(targets, predictions)
    cohen_kappa = cohen_kappa_score(targets, predictions)

    metrics = {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "mcc": float(mcc),
        "cohen_kappa": float(cohen_kappa),
        "evaluation_seconds": float(evaluation_seconds),
    }

    return metrics, targets, predictions


In [ ]:
# Cell 18: Train FedProx rounds, evaluate after every round, and checkpoint
round_history = []
client_history_records = []
start_round = 1

if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    saved_cfg = checkpoint.get("config", {})

    expected_resume_values = {
        "seed": SEED,
        "num_clients": NUM_CLIENTS,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "proximal_mu": PROXIMAL_MU,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    }
    for key, expected in expected_resume_values.items():
        if key in saved_cfg and saved_cfg[key] != expected:
            raise ValueError(
                f"Checkpoint configuration mismatch for {key}: "
                f"checkpoint={saved_cfg[key]} current={expected}. "
                "Delete/rename the old checkpoint or restore the previous configuration."
            )

    global_model.load_state_dict(checkpoint["global_model_state_dict"])
    round_history = checkpoint.get("round_history", [])
    client_history_records = checkpoint.get("client_history", [])
    start_round = int(checkpoint.get("round", 0)) + 1
    print(f"Resuming from completed FL round {start_round - 1}.")

training_session_start = time.perf_counter()

for round_number in range(start_round, NUM_ROUNDS + 1):
    round_summary, client_df = run_fedprox_round(round_number)

    print(f"\nEvaluating global model after round {round_number}...")
    metrics, _, _ = evaluate_global_model(global_model)
    round_summary.update(metrics)
    round_history.append(round_summary)
    client_history_records.extend(client_df.to_dict(orient="records"))

    torch.save({
        "round": round_number,
        "global_model_state_dict": global_model.state_dict(),
        "round_history": round_history,
        "client_history": client_history_records,
        "config": {
            "strategy": "FedProx",
            "num_rounds": NUM_ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "proximal_mu": PROXIMAL_MU,
            "seed": SEED,
            "num_clients": NUM_CLIENTS,
            "partition": "Dirichlet label-skew",
            "dirichlet_alpha": DIRICHLET_ALPHA,
        },
        "classes": classes,
        "feature_cols": feature_cols,
        "prepared_metadata": metadata,
    }, CHECKPOINT_PATH)

    print("\nRound summary:")
    for key, value in round_summary.items():
        if isinstance(value, float):
            print(f"{key:<32} {value:.6f}")
        else:
            print(f"{key:<32} {value}")
    print("Checkpoint saved:", CHECKPOINT_PATH)

training_session_seconds = time.perf_counter() - training_session_start
round_history_df = pd.DataFrame(round_history)
client_history_df = pd.DataFrame(client_history_records)

print("\nFederated round history:")
display(round_history_df)
print("This session training time:", f"{training_session_seconds:.2f} s")


In [ ]:
# Cell 19: Final global evaluation and per-class metrics
final_metrics, test_targets, test_predictions = evaluate_global_model(global_model)

print("Final global-model metrics:")
for name, value in final_metrics.items():
    if name != "evaluation_seconds":
        print(f"{name:<24} {value:.4f}")
print("Evaluation time:", f"{final_metrics['evaluation_seconds']:.2f} s")

report_dict = classification_report(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

per_class_df = pd.DataFrame([
    {
        "class_id": i,
        "class_name": name,
        "precision": report_dict[name]["precision"],
        "recall": report_dict[name]["recall"],
        "f1": report_dict[name]["f1-score"],
        "support": int(report_dict[name]["support"]),
    }
    for i, name in enumerate(classes)
])

display(per_class_df.sort_values("f1"))


In [ ]:
# Cell 20: Convergence plots
if not round_history_df.empty:
    plt.figure(figsize=(7, 4))
    plt.plot(
        round_history_df["round"],
        round_history_df["client_train_loss"],
        marker="o",
        label="CE loss",
    )
    plt.plot(
        round_history_df["round"],
        round_history_df["client_objective_loss"],
        marker="o",
        label="FedProx objective",
    )
    plt.xlabel("Federated round")
    plt.ylabel("Weighted client loss")
    plt.title("FedProx Non-IID — Local Training Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fedprox_non_iid_train_loss_by_round.png", dpi=160)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(
        round_history_df["round"],
        round_history_df["macro_f1"],
        marker="o",
        label="Macro F1",
    )
    plt.plot(
        round_history_df["round"],
        round_history_df["accuracy"],
        marker="o",
        label="Accuracy",
    )
    plt.xlabel("Federated round")
    plt.ylabel("Score")
    plt.title("FedProx Non-IID — Global Test Performance")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fedprox_non_iid_global_metrics_by_round.png", dpi=160)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(
        round_history_df["round"],
        round_history_df["client_train_accuracy_std"],
        marker="o",
    )
    plt.xlabel("Federated round")
    plt.ylabel("Std. dev. of client training accuracy")
    plt.title("FedProx Non-IID — Client Training Dispersion")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "fedprox_non_iid_client_accuracy_dispersion.png",
        dpi=160,
    )
    plt.show()


In [ ]:
# Cell 21: Normalized confusion matrix — same diagnostic style as previous runs
cm_norm = confusion_matrix(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    normalize="true",
)

plt.figure(figsize=(18, 16))
plt.imshow(cm_norm, interpolation="nearest", aspect="auto")
plt.colorbar()
plt.xticks(np.arange(len(classes)), classes, rotation=90, fontsize=7)
plt.yticks(np.arange(len(classes)), classes, fontsize=7)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("FedProx Non-IID — Normalized Confusion Matrix")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "fedprox_non_iid_confusion_matrix_normalized.png",
    dpi=180,
)
plt.show()


In [ ]:
# Cell 22: Comparison-ready result row and experiment summary
final_round = int(round_history_df["round"].max()) if not round_history_df.empty else 0

total_round_training_seconds = (
    float(round_history_df["round_training_seconds"].sum())
    if not round_history_df.empty else 0.0
)

client_size_cv = float(client_sizes.std(ddof=0) / client_sizes.mean())
mean_js = float(partition_summary_df["js_divergence_from_global"].mean())
mean_dominant_share = float(
    partition_summary_df["dominant_class_share"].mean()
)
mean_active_classes = float(partition_summary_df["active_classes"].mean())

# Core metric columns intentionally match the earlier comparison CSVs.
comparison_metrics_df = pd.DataFrame([{
    "experiment": "FedProx-Non-IID",
    "dataset_frac": float(metadata["dataset_frac_requested"]),
    "train_samples": int(len(y_train_t)),
    "test_samples": int(len(y_test_t)),
    "num_clients": NUM_CLIENTS,
    "rounds": final_round,
    "local_epochs": LOCAL_EPOCHS,
    "effective_data_passes": final_round * LOCAL_EPOCHS,
    "accuracy": final_metrics["accuracy"],
    "balanced_accuracy": final_metrics["balanced_accuracy"],
    "macro_precision": final_metrics["macro_precision"],
    "macro_recall": final_metrics["macro_recall"],
    "macro_f1": final_metrics["macro_f1"],
    "weighted_f1": final_metrics["weighted_f1"],
    "mcc": final_metrics["mcc"],
    "cohen_kappa": final_metrics["cohen_kappa"],
    "partition": "Dirichlet-label-skew",
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "proximal_mu": PROXIMAL_MU,
    "client_size_min": int(client_sizes.min()),
    "client_size_max": int(client_sizes.max()),
    "client_size_cv": client_size_cv,
    "mean_active_classes": mean_active_classes,
    "mean_dominant_class_share": mean_dominant_share,
    "mean_js_divergence_from_global": mean_js,
}])

display(comparison_metrics_df.round(4))

experiment_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "federated_strategy": "FedProx",
    "proximal_mu": PROXIMAL_MU,
    "partition": {
        "type": "Dirichlet label-skew",
        "alpha": DIRICHLET_ALPHA,
        "seed": SEED,
        "attempts": partition_attempts,
        "min_client_samples_constraint": MIN_CLIENT_SAMPLES,
        "client_sizes": [int(x) for x in client_sizes],
        "client_size_cv": client_size_cv,
        "mean_active_classes": mean_active_classes,
        "mean_dominant_class_share": mean_dominant_share,
        "mean_js_divergence_from_global": mean_js,
    },
    "prepared_artifact": metadata,
    "model": {
        "architecture": f"{len(feature_cols)}-{HIDDEN_1}-{HIDDEN_2}-{len(classes)}",
        "trainable_parameters": int(num_parameters),
    },
    "training": {
        "num_clients": NUM_CLIENTS,
        "rounds": final_round,
        "local_epochs": LOCAL_EPOCHS,
        "effective_data_passes": final_round * LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "proximal_mu": PROXIMAL_MU,
        "all_clients_per_round": True,
        "use_amp": amp_enabled,
        "training_data_location": str(train_data_device),
    },
    "metrics": {k: float(v) for k, v in final_metrics.items()},
    "round_history": round_history,
    "runtime_seconds": {
        "materialize_and_scale": float(materialize_seconds),
        "partition_creation": float(partition_seconds),
        "round_training_total": float(total_round_training_seconds),
        "this_session": float(training_session_seconds),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    },
}


In [ ]:
# Cell 23: Save all comparison/report artifacts
results_path = OUTPUT_DIR / "fedprox_non_iid_results.json"
round_history_path = OUTPUT_DIR / "fedprox_non_iid_round_history.csv"
client_history_path = OUTPUT_DIR / "fedprox_non_iid_client_history.csv"
per_class_path = OUTPUT_DIR / "fedprox_non_iid_per_class.csv"
comparison_path = OUTPUT_DIR / "fedprox_non_iid_comparison_metrics.csv"
partition_summary_path = OUTPUT_DIR / "fedprox_non_iid_partition_summary.csv"
class_distribution_path = (
    OUTPUT_DIR / "fedprox_non_iid_client_class_distribution.csv"
)
partition_npz_path = OUTPUT_DIR / "fedprox_non_iid_client_positions.npz"

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, indent=2)

round_history_df.to_csv(round_history_path, index=False)
client_history_df.to_csv(client_history_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
comparison_metrics_df.to_csv(comparison_path, index=False)
partition_summary_df.to_csv(partition_summary_path, index=False)
client_class_distribution_df.to_csv(class_distribution_path, index=False)

# Save the exact deterministic split for reproducibility.
np.savez_compressed(
    partition_npz_path,
    **{
        f"client_{i:02d}": positions
        for i, positions in enumerate(client_positions)
    },
)

print("Saved result artifacts:")
for path in [
    CHECKPOINT_PATH,
    results_path,
    round_history_path,
    client_history_path,
    per_class_path,
    comparison_path,
    partition_summary_path,
    class_distribution_path,
    partition_npz_path,
]:
    print("-", path)


In [ ]:
# Cell 24: Package result outputs for download
# The multi-GB prepared dataset is not included.
zip_path = Path("/kaggle/working/fedprox_non_iid_all_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.iterdir():
        if file.is_file():
            z.write(file, arcname=file.name)

print("Created:", zip_path)
print("Zip size:", f"{zip_path.stat().st_size / 1024**2:.2f} MB")


## What to keep for the final comparison

After this notebook completes, keep:

- `fedprox_non_iid_comparison_metrics.csv` — final comparison-ready summary.
- `fedprox_non_iid_round_history.csv` — convergence and per-round global metrics.
- `fedprox_non_iid_client_history.csv` — local CE loss, FedProx objective, proximal penalty, accuracy, and runtime for each client.
- `fedprox_non_iid_per_class.csv` — per-class precision/recall/F1/support.
- `fedprox_non_iid_partition_summary.csv` — quantitative non-IID descriptors.
- `fedprox_non_iid_client_positions.npz` — exact partition used.
- `fedprox_non_iid_checkpoint.pt` — final global model plus training history.
- `fedprox_non_iid_all_outputs.zip` — packaged experiment outputs.

### Fair comparison

For the direct comparison with `FedAvg-Non-IID`, keep all settings identical. The intended changed variable is the **local optimization rule**:

- FedAvg: cross-entropy local objective
- FedProx: cross-entropy + proximal penalty

The notebook uses `PROXIMAL_MU = 0.01` and records it explicitly in the outputs.
